<a href="https://colab.research.google.com/github/warllemedicao/Extensao-Estacio/blob/main/run_colab_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Treinamento Neural de Voz - Pipeline automatico no Colab

Use **Executar tudo**. O notebook monta o Drive, baixa/atualiza o projeto, instala dependencias, copia os audios e executa limpeza, preprocessamento, alinhamento MFA, organizacao FastSpeech2 e treino.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/warllemedicao/Voz_neural.git"
REPO_DIR = Path("/content/Voz_neural")
PROJECT_DIR = REPO_DIR / "Treinamento neural"

if REPO_DIR.exists():
    print(f"Projeto ja existe em {REPO_DIR}. Tentando atualizar...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

if not PROJECT_DIR.exists():
    raise RuntimeError(f"Pasta do projeto nao encontrada: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"Diretorio ativo: {PROJECT_DIR}")

Diretorio ativo: /content/Voz_neural/Treinamento neural


In [5]:
import subprocess
import sys

def run_checked(cmd, cwd=None):
    print("\n$ " + " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, check=True)

run_checked(["apt-get", "update"])
run_checked(["apt-get", "install", "-y", "sox", "libsndfile1", "ffmpeg"])

run_checked([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchaudio", "torchvision",
    "librosa", "pyworld", "pyyaml", "tqdm", "matplotlib", "scipy",
    "tensorboard", "soundfile", "numpy", "tgt",
    "openai-whisper", "demucs", "montreal-forced-aligner"
])


$ apt-get update

$ apt-get install -y sox libsndfile1 ffmpeg

$ /usr/bin/python3 -m pip install -q torch torchaudio torchvision librosa pyworld pyyaml tqdm matplotlib scipy tensorboard soundfile numpy tgt openai-whisper demucs montreal-forced-aligner


In [6]:
from pathlib import Path
import shutil

drive_candidates = [
    Path("/content/drive/MyDrive/Treinamento_Voz/Audios_Brutos"),
    Path("/content/drive/MyDrive/Audios_Brutos"),
]
input_dir = next((p for p in drive_candidates if p.exists()), None)
colab_input = PROJECT_DIR / "Audios_brutos"
colab_input.mkdir(parents=True, exist_ok=True)

if input_dir is None:
    tried = "\n".join(str(p) for p in drive_candidates)
    raise FileNotFoundError(f"Pasta de audios nao encontrada. Caminhos testados:\n{tried}")

audio_exts = {".mp3", ".wav", ".ogg", ".m4a"}
copied = 0
for src in sorted(input_dir.rglob("*")):
    if src.is_file() and src.suffix.lower() in audio_exts:
        dst = colab_input / src.name
        shutil.copy2(src, dst)
        copied += 1

if copied == 0:
    raise RuntimeError(f"Nenhum audio valido encontrado em {input_dir}")

print(f"Copiados {copied} audio(s) de {input_dir} para {colab_input}")

Copiados 522 audio(s) de /content/drive/MyDrive/Treinamento_Voz/Audios_Brutos para /content/Voz_neural/Treinamento neural/Audios_brutos


## Execucao automatica

A celula abaixo para imediatamente se uma etapa falhar, mostrando qual comando quebrou.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import yaml
import torch

os.chdir(PROJECT_DIR)
processed_audio = PROJECT_DIR / "Audios_processados"
preprocessed = PROJECT_DIR / "preprocessed_data"
fastspeech_dir = PROJECT_DIR / "fastspeech2"

def save_yaml(path, data):
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

preprocess_yaml = PROJECT_DIR / "voz_final" / "config" / "LJSpeech" / "preprocess.yaml"
with open(preprocess_yaml, "r", encoding="utf-8") as f:
    preprocess_cfg = yaml.safe_load(f)
preprocess_cfg["path"]["source_path"] = "Audios_processados"
preprocess_cfg["path"]["corpus_path"] = "preprocessed_data/corpus"
preprocess_cfg["path"]["preprocessed_path"] = "preprocessed_data"
save_yaml(preprocess_yaml, preprocess_cfg)

ljspeech_yaml = fastspeech_dir / "config" / "LJSpeech" / "ljspeech.yaml"
with open(ljspeech_yaml, "r", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)
train_cfg["path"]["corpus_path"] = "../preprocessed_data"
train_cfg["path"]["lexicon_path"] = "../lexicon/ljspeech-lexicon.txt"
train_cfg["device"] = "cuda" if torch.cuda.is_available() else "cpu"
save_yaml(ljspeech_yaml, train_cfg)

def run_step(name, cmd, cwd):
    print("\n" + "=" * 80)
    print(name)
    print("$ " + " ".join(map(str, cmd)))
    print("cwd:", cwd)
    print("=" * 80)
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Etapa falhou ({name}) com codigo {code}")

run_step(
    "[1/5] Limpeza e transcricao",
    [sys.executable, "limpeza_ia.py", "--input_dir", "Audios_brutos", "--output_dir", "Audios_processados"],
    PROJECT_DIR,
)

train_txt = processed_audio / "train.txt"
if not train_txt.exists() or not train_txt.read_text(encoding="utf-8").strip():
    raise RuntimeError(f"train.txt nao foi gerado corretamente: {train_txt}")

run_step(
    "[2/5] Preprocessamento voz_final + MFA",
    [sys.executable, "preprocess_full.py"],
    PROJECT_DIR / "voz_final",
)

duration_dir = preprocessed / "duration"
if not duration_dir.exists() or not list(duration_dir.glob("*.npy")):
    raise RuntimeError(f"Nenhuma duration MFA foi gerada em {duration_dir}")

run_step(
    "[3/5] Organizacao do dataset FastSpeech2",
    [sys.executable, "preprocess.py"],
    fastspeech_dir,
)

processed_txt = preprocessed / "processed" / "txt" / "train"
if not processed_txt.exists() or not list(processed_txt.glob("*.txt")):
    raise RuntimeError(f"Dataset FastSpeech2 nao foi organizado em {processed_txt}")

run_step(
    "[4/5] Treinamento FastSpeech2",
    [sys.executable, "train.py"],
    fastspeech_dir,
)

print("\n[5/5] Pipeline completo finalizado.")


[1/5] Limpeza e transcricao
$ /usr/bin/python3 limpeza_ia.py --input_dir Audios_brutos --output_dir Audios_processados
cwd: /content/Voz_neural/Treinamento neural
[INFO] Ambiente detectado: colab
 🎤 LIMPEZA DE ÁUDIO COM ANÁLISE INTELIGENTE
Entrada: /content/Voz_neural/Treinamento neural/Audios_brutos
Saída: /content/Voz_neural/Treinamento neural/Audios_processados
Arquivos encontrados: 522
Modo: Normal (usar cache)
Ambiente: colab

[INFO] Carregando modelo Whisper para transcrição...

  0%|                                              | 0.00/1.42G [00:00<?, ?iB/s]
  0%|                                     | 3.80M/1.42G [00:00<00:38, 39.9MiB/s]
  2%|▊                                     | 29.5M/1.42G [00:00<00:08, 175MiB/s]
  3%|█▏                                    | 46.8M/1.42G [00:00<00:08, 178MiB/s]
  5%|█▋                                    | 65.6M/1.42G [00:00<00:07, 185MiB/s]
  6%|██▏                                   | 84.8M/1.42G [00:00<00:07, 191MiB/s]
  7%|██▊               